In [1]:
from IPython.display import display, HTML
display(HTML("""
<style>
div.container{width:95% !important;}
div.cell.code_cell.rendered{width:100%;}
div.CodeMirror {font-family:Consolas; font-size:15pt;}
div.output {font-size:15pt; font-weight:bold;}
div.input {font-family:Consolas; font-size:15pt;}
div.prompt {min-width:70px;}
div#toc-wrapper{padding-top:120px;}
div.text_cell_render ul li{font-size:12pt;padding:5px;}
table.dataframe{font-size:15px;}
</style>
"""))

[ RAG 구현 절차 ]

    1.	문서의 내용을 읽는다(document_loader를 이용)
    (1)	https://python.langchain.com/v0.2/docs/integrations/document_loaders/ 
    (2)	https://python.langchain.com/v0.2/docs/integrations/document_loaders/microsoft_word/
    %pip install --upgrade --quiet  docx2txt
    2.	문서를 쪼갠다(한번에 이해하고 처리할 수 있는 입력+출력 토큰수가 제한)
    (1)	 https://python.langchain.com/v0.2/docs/how_to/recursive_text_splitter/#splitting-text-from-languages-without-word-boundaries 
    %pip install -qU langchain-text-splitters
    3.	쪼갠 문서를 임베딩하여 vector database에 넣음
    (1)	OpenAIEmbeddings나 UpstageEmbeddings이용해서 임베딩
    (2)	https://python.langchain.com/v0.2/docs/integrations/vectorstores/chroma/  
    %pip install –q langchain-chroma
    4.	질문을 이용해 유사도 검색
    5.	유사도 검색한 문서를 LLM에 질문으로 전달하여 답변 얻음(제공되는 Prompt활용)
    (1)	https://python.langchain.com/v0.2/docs/tutorials/rag/
    %pip install –q langchain langchainhub
    http://smith.langchain.com에서 key생성 .env key(LANGCHAIN_API_KEY) 추가

## 2. 문서를 쪼개면서 읽기(o)

In [1]:
import time
start = time.time()
from langchain_community.document_loaders import Docx2txtLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
loader = Docx2txtLoader('./tax_docs/소득세법(법률)(제20615호)(20250701).docx')
text_splitter = RecursiveCharacterTextSplitter(  # 문서를 쪼개는 기준이 문자수
    chunk_size=1500, #문서를 쪼갤때 1500글자씩 쪼개
    chunk_overlap=200
)
# 1번째 chunk 1~1450글자
# 2번째 chunk 1250~1750글자
documents = loader.load_and_split(text_splitter=text_splitter)
runtime = time.time() - start
print('문서 쪼개면서 읽는 시간 :', runtime)

문서 쪼개면서 읽는 시간 : 5.918393135070801


## 3. 쪼갠문서를 임베딩 -> 벡터 데이터베이스 저장
- 임베딩 모델 : upstage의 solar-embedding-1-large
- 벡터 데이터베이스 : chroma

In [2]:
# https://python.langchain.com/v0.2/docs/integrations/text_embedding/upstage
from dotenv import load_dotenv
from langchain_upstage import UpstageEmbeddings
load_dotenv()
embeddings = UpstageEmbeddings(
    model="solar-embedding-1-large"
    # model="embedding-query"
)

In [3]:
doc_result = embeddings.embed_documents(
    ["소득세법 어쩌구 저쩌구", documents[0].page_content]
)
print(len(doc_result), len(doc_result[0]), len(doc_result[1]))

2 4096 4096


In [4]:
%%time
from langchain_chroma import Chroma
# 데이터를 처음 저장할 때
# database = Chroma.from_documents(                                 
#     documents=documents,
#     embedding=embeddings,
#     collection_name="tax-collection", # 생략시 이름 랜덤
#     persist_directory='./chroma_upstage'      # 생략시 로컬데이터베이스에 저장안됨. 프로그램 종료시 db날라감
# )
# 이미 저장된 vector DB를 사용할 때
database = Chroma(
    embedding_function=embeddings,
    collection_name="tax-collection",
    persist_directory='./chroma_upstage'
)

CPU times: total: 906 ms
Wall time: 1.09 s


## 4. vector DB에 질문과 유사도 검색(답변 생성을 위한 retrieval)

In [6]:
query = "연봉 5천만원인 직장인의 소득세는 얼마인가요?"
retrieved_docs = database.similarity_search(query,
                                           k=3) # 기본 k는 4

In [7]:
context_text = "\n\n".join([doc.page_content for doc in retrieved_docs])

In [9]:
retrieved_docs

[]

## 5. 유사도 검색으로 가져온 문서를 질문과 같이 LLM 전달하여 답변 생성

In [9]:
from langchain_openai import ChatOpenAI
llm = ChatOpenAI(model="gpt-4.1-nano")

In [10]:
prompt = f"""[identity]
- 당신은 최고의 한국 소득세 전문자입니다
- [context]를 참고해서 사용자의 질문에 답변해 주세요
[context]는 다음과 같아요
{retrieved_docs}
Question : {query}"""

In [11]:
ai_message = llm.invoke(prompt)

In [12]:
print(ai_message.content)

연봉이 5천만 원인 직장인의 소득세를 계산하기 위해서는 지방소득세를 포함한 세액을 추산해야 합니다. 상세 계산 과정은 다음과 같습니다:

1. **근로소득공제 계산:**
- 연봉이 5,000만 원인 경우, 근로소득공제는 대략 1,206만 원입니다.  
(근로소득공제액은 소득 구간에 따라 다르며, 5,000만 원 기준 약 1,206만 원입니다.)

2. **과세표준 계산:**  
연봉 - 근로소득공제 = 5,000만 원 - 1,206만 원 = 약 3,794만 원

3. **소득세율 적용:**  
과세표준 3,794만 원에 대해 아래 세율을 적용합니다:

- 1,200만 원 이하: 6%  
- 1,200만 원 초과 ~ 4,600만 원 이하: 15%

계산 방식은 누진세율이므로 각 구간별 세율을 적용하여 구합니다:

- 1,200만 원까지: 1,200만 원 × 6% = 72만 원
- 나머지: 3,794만 원 - 1,200만 원 = 2,594만 원  
  → 2,594만 원 × 15% = 약 389.1만 원

**소득세 합계:**  
72만 원 + 389.1만 원 ≈ 461.1만 원

4. **지방소득세:**  
소득세의 10%인 지방소득세가 추가로 부과되므로  
약 46만 원이 더 발생합니다.

---

### 최종 세액 (근로소득세 + 지방소득세):  
약 **507만 원**

---

**참고:**  
이 계산은 기본 공제(인적공제, 특별공제 등)를 고려하지 않은 대략적인 금액입니다. 실제 세액은 공제 항목에 따라 달라질 수 있으며, 세법 개정 사항에 따라 변동될 수 있으니 정확한 계산을 위해 세무 전문가와 상담하시기 바랍니다.


## 5. Augmentation을 위한 제공되는 Prompt활용하여 langchain으로 답변 생성

In [13]:
query = "연봉 5천만원인 직장인의 소득세는 얼마인가요?"

from langchain import hub
prompt = hub.pull("rlm/rag-prompt")
prompt

C:\Users\Admin\anaconda3\envs\LLM\lib\site-packages\langsmith\client.py:272: LangSmithMissingAPIKeyWarning: API key must be provided when using hosted LangSmith API
  warnings.warn(


ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, metadata={'lc_hub_owner': 'rlm', 'lc_hub_repo': 'rag-prompt', 'lc_hub_commit_hash': '50442af133e61576e74536c6556cefe1fac147cad032f4377b60c436e6cdcb6e'}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template="You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say that you don't know. Use three sentences maximum and keep the answer concise.\nQuestion: {question} \nContext: {context} \nAnswer:"), additional_kwargs={})])

### RetrievalQA를 통해 LLM전달 (create_retrieval_chain이 대체)
     query -> retriever전달(백터 검색 수행) 
     -> retrieval문서 -> prompt의 {context}에 삽입
     -> query -> prompt의 {question}에 삽입

In [14]:
from langchain.chains import RetrievalQA
qa_chain = RetrievalQA.from_chain_type(
    llm,
    retriever = database.as_retriever(search_kwargs={'k':5}),
    chain_type_kwargs={"prompt":prompt}
)

In [15]:
ai_message = qa_chain.invoke({"query":query})

In [16]:
ai_message

{'query': '연봉 5천만원인 직장인의 소득세는 얼마인가요?',
 'result': '연봉 5천만원인 직장인의 소득세는 여러 요소에 따라 다르지만, 일반적으로 소득세는 과세표준과 세율에 따라 계산됩니다. 대한민국의 소득세율은 누진세 구조로 최대 45%까지 적용되며, 공제와 세액공제도 고려되어야 합니다. 따라서 정확한 금액은 연말 소득세 신고 시 구체적인 세액 계산이 필요합니다.'}